Geração e Recomendação de Trajetórias Sintéticas de Transporte Público
 Este *notebook* apresenta o *pipeline* completo de modelagem de rotas de ônibus utilizando o motor probabilístico `Palmto-gen`. O objetivo é ler dados de GPS com ruídos, transformar o mapa urbano em um sistema de *tokens* (grid espacial), aprender o comportamento espacial dos veículos via Trigramas e gerar novas trajetórias sintéticas consistentes. Por fim, o projeto culmina em um Sistema de Recomendação de Rotas com visualização interativa.
 
Arquivos Necessários para Execução
 Para que este código rode corretamente, você precisa garantir que os seguintes arquivos estejam nos caminhos referenciados nas variáveis do código:
 1. Banco de Trajetórias (`rota328.pkl` ou similar): Arquivo contendo os dados pré-processados e limpos da frota. Ele deve possuir as colunas `trip_id` e `geometry` (com pares de coordenadas geográficas).
 2. Malha Geográfica (`Centro.geojson` ou similar): Um polígono contendo o limite oficial da região de estudo. Ele é fundamental para que o algoritmo saiba onde recortar o mapa em células.

In [ ]:
!pip install Palmto-gen
!pip install pyarrow
!pip install matplotlib

1. Ingestão de Dados
 Nesta etapa, importamos o arquivo `.pkl` gerado na fase de pré-processamento (onde "pulos" de GPS foram removidos). Utilizamos a biblioteca `pandas` para estruturar essas geometrias.

**Load Data:**  The Palmto_gen package has a sample file of 30k taxi trajectories from the city of Porto included that we can use for demonstration.

In [ ]:
import pandas as pd
from importlib.resources import files

sample_data_path = files('Palmto_gen').joinpath('/home/maria_araujo/grupo2/.venv/lib/python3.12/site-packages/Palmto_gen/data/rota328.pkl')

df = pd.read_pickle(sample_data_path)
df.head()

Next, we convert the trajectories into *'sentences'* using the following steps:
1. **Create Shapely Points:** For each latitude-longitude coordinate pair, create a Shapely Point object.
2. **Overlay a Grid**: Overlay a grid of a specific size over the area covered by the trajectories.
3. Assign Unique IDs to Grid Cells
4. **Merge the shapely points with the cell:** representing each point by the ID of the cell it fall into.

2. Transformando Coordenadas em Tokens
 Para que o modelo probabilístico funcione, ele precisa entender o mapa não como números contínuos, mas como "palavras". O processo abaixo executa as seguintes etapas lógicas:
 1. Criação de Pontos (`Shapely Points`): Converte a lista de `[lat, lon]` em objetos geométricos matemáticos.
 2. Geração do Grid: Sobrepõe uma grade (grid) de células de 50 metros por 50 metros em cima do polígono do GeoJSON.
 3. Indexação: Atribui um ID único para cada célula (como um tabuleiro de xadrez).
 4. Merge: Alinha cada ponto de GPS à célula correspondente, transformando a viagem do ônibus em uma "frase" (sequência de IDs).

In [ ]:
import geopandas as gpd
from Palmto_gen import ConvertToToken
from importlib.resources import files

# Substituindo o pkg_resources pela abordagem moderna
study_area_path = files('Palmto_gen').joinpath('/home/maria_araujo/grupo2/.venv/lib/python3.12/site-packages/Palmto_gen/data/Centro.geojson')

# O GeoPandas também consegue ler direto do caminho gerado
study_area = gpd.read_file(study_area_path)

# Executando o resto do seu código normalmente
TokenCreator = ConvertToToken(df, study_area, cell_size=50)
grid, sentence_df = TokenCreator.create_tokens()

# Mostrando os resultados
print(sentence_df.head())

**Create n-grams:** from the *'sentences'* we formed in the previous step.

3. Extração de Trigramas (O Cérebro do Modelo)
 Com as "frases" formadas, instanciamos o `NgramGenerator`. Ele mapeia todas as viagens reais e aprende a probabilidade de transição.

In [ ]:
from Palmto_gen import NgramGenerator

ngram_model = NgramGenerator(sentence_df)
ngrams, start_end_points = ngram_model.create_ngrams()

**Approach 1: Generating length-constrained trajectories
from a given point**

Start the process by selecting a token from
our bigram corpus and specifying the length (number of points) for
our trajectory and the number of new trajectories we want. The model then proceeds to construct the trajectory by iteratively generating and adding points
until the predetermined length is reached.

4. Geração de Novas Trajetórias
 Abordagem 1: Rotas restritas por Comprimento
 A partir de um ponto de origem aleatório conhecido, o modelo gera novas trajetórias sintéticas prevendo os próximos passos de forma autônoma.

In [ ]:
from Palmto_gen import TrajGenerator

n = 10000
sentence_length = 30
traj_generator = TrajGenerator(ngrams, start_end_points, n, grid)
new_trajs_app1, new_trajs_app1_gdf = traj_generator.generate_trajs_using_origin(sentence_length, seed=None)
new_trajs_app1.head()

**Approach 2: Generating trajectories between two given
points**

Select one origin and one destination point
from one of our original trajectories and generate trajectories that
connect these two points.

Abordagem 2: Rotas Guiadas por Origem e Destino
 Conecta um ponto inicial e final predeterminados, simulando caminhos que ligam as duas pontas respeitando a malha probabilística.

In [ ]:
n = 10000
traj_generator = TrajGenerator(ngrams, start_end_points, n, grid)
new_trajs_app2, new_trajs_app2_gdf = traj_generator.generate_trajs_using_origin_destination()
new_trajs_app2.head()

Salvar o novo dataset gerado

**Save the new dataset.**

In [ ]:
new_trajs_app1.to_pickle('generated_trajs_app1.pkl')
new_trajs_app1.to_csv('generated_trajs_app1.csv')

**Plot**: a sample of 1000 trajectories from the original dataset and the dataset generated using approach 1.

5. Validação Visual
 Plotamos uma amostra de trajetórias reais e sintéticas (Abordagem 1) para visualização através de mapas interativos Folium.

In [ ]:
from Palmto_gen import DisplayTrajs

# Calcula o tamanho máximo da amostra
tamanho_amostra_original = min(1000, len(sentence_df))
tamanho_amostra_gerado = min(1000, len(new_trajs_app1_gdf))

original_trajs = sentence_df['geometry'].sample(tamanho_amostra_original).to_list()
generated_trajs = new_trajs_app1_gdf['geometry'].sample(tamanho_amostra_gerado).to_list()

# Inicia o gerador de mapas
display_trajs = DisplayTrajs(original_trajs, generated_trajs)

# PLANO B: Em vez de tentar mostrar no VS Code, criamos os mapas separadamente
mapa_original = display_trajs.plot_map(original_trajs)
mapa_gerado = display_trajs.plot_map(generated_trajs)

# Salvamos como arquivos de site (.html)
mapa_original.save("mapa_original.html")
mapa_gerado.save("mapa_gerado.html")

print("Mapas gerados! Vá na pasta do seu projeto e dê um duplo clique nos arquivos .html")

**Plot**: a sample of 1000 trajectories from the original dataset and the dataset generated using approach 2.


Plotando a Abordagem 2.

In [ ]:
from Palmto_gen import DisplayTrajs

# 1. Trava de segurança para não pedir mais trajetos do que o banco possui
tamanho_amostra_original = min(1000, len(sentence_df))
tamanho_amostra_gerado = min(1000, len(new_trajs_app2_gdf))

original_trajs = sentence_df['geometry'].sample(tamanho_amostra_original).to_list()
generated_trajs = new_trajs_app2_gdf['geometry'].sample(tamanho_amostra_gerado).to_list()

# 2. Inicia o construtor do mapa
display_trajs = DisplayTrajs(original_trajs, generated_trajs)

# 3. Gerando e salvando os mapas em HTML (Plano B)
mapa_original_app2 = display_trajs.plot_map(original_trajs)
mapa_gerado_app2 = display_trajs.plot_map(generated_trajs)

mapa_original_app2.save("mapa_original_app2.html")
mapa_gerado_app2.save("mapa_gerado_app2.html")

print("Mapas da Abordagem 2 gerados com sucesso! Verifique sua pasta.")

**Heat Map**: comparing the original trajectories with the trajectories generated using approach 1.

Mapas de Calor (Heat Maps)
 Comparamos as densidades espaciais para avaliar se o modelo preservou as características estruturais originais (Abordagem 1).

In [ ]:
import matplotlib.pyplot as plt

# 1. Trava inteligente para o mapa de calor
tamanho_amostra_original = min(5000, len(sentence_df))
tamanho_amostra_gerado1 = min(5000, len(new_trajs_app1_gdf))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 2. Usa a trava dentro do sample()
display_trajs.plot_heat_map(sentence_df.sample(tamanho_amostra_original), study_area, axes[0], cell_size = 800)
axes[0].set_title('Original Trajectories')

display_trajs.plot_heat_map(new_trajs_app1_gdf.sample(tamanho_amostra_gerado1), study_area, axes[1], cell_size = 800)
axes[1].set_title('Generated Trajectories')

plt.tight_layout()
plt.show()

**Heat Map**: comparing the original trajectories with the trajectories generated using approach 2.

Mapas de Calor (Abordagem 2)

In [ ]:
import matplotlib.pyplot as plt

# 1. Trava inteligente
tamanho_amostra_original = min(5000, len(sentence_df))
tamanho_amostra_gerado2 = min(5000, len(new_trajs_app2_gdf))

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# 2. Usa a trava dentro do sample()
display_trajs.plot_heat_map(sentence_df.sample(tamanho_amostra_original), study_area, axes[0], cell_size = 800)
axes[0].set_title('Original Trajectories')

display_trajs.plot_heat_map(new_trajs_app2_gdf.sample(tamanho_amostra_gerado2), study_area, axes[1], cell_size = 800)
axes[1].set_title('Generated Trajectories')

plt.tight_layout()
plt.show()

# ## 6. Sistema de Recomendação de Rotas (Aplicação Final)
# A classe abaixo implementa geofencing para filtrar e recomendar a melhor rota entre dois pontos geográficos específicos, classificando a opção mais rápida (menor distância real) utilizando os dados que o projeto estruturou.

In [ ]:
import pandas as pd
import folium
from geopy.distance import geodesic

class RecomendadorDeRotas:
    # ATENÇÃO AQUI: São DOIS underlines de cada lado! -> __init__
    def __init__(self, arquivo_pkl):
        self.df_rotas = pd.read_pickle(arquivo_pkl)
        self.cores_alternativas = ['#8A2BE2', '#FF8C00', '#1E90FF', '#FF1493', '#00CED1']
        self.cor_melhor_rota = '#32CD32' 

        self.locais = {
            "Ilha do Governador": (-22.805, -43.204),
            "Bonsucesso": (-22.864, -43.255),
            "Fundão (UFRJ)": (-22.858, -43.232),
            "Centro": (-22.906, -43.177)
        }

    def calcular_distancia_rota(self, pontos):
        distancia_total = 0
        for i in range(len(pontos) - 1):
            p1 = (pontos[i][1], pontos[i][0])
            p2 = (pontos[i+1][1], pontos[i+1][0])
            distancia_total += geodesic(p1, p2).km
        return distancia_total

    def analisar_rotas_com_filtro(self, local_origem, local_destino, raio_km=2.0):
        coord_origem = self.locais[local_origem]
        coord_destino = self.locais[local_destino]
        
        resultados = []
        
        for index, row in self.df_rotas.iterrows():
            id_rota = row['trip_id']
            pontos = row['geometry']
            
            if not pontos or len(pontos) < 5: 
                continue

            passou_origem = False
            indice_origem = -1
            
            for i, pt in enumerate(pontos):
                if geodesic((pt[1], pt[0]), coord_origem).km <= raio_km:
                    passou_origem = True
                    indice_origem = i
                    break
            
            passou_destino = False
            if passou_origem:
                for pt in pontos[indice_origem:]:
                    if geodesic((pt[1], pt[0]), coord_destino).km <= raio_km:
                        passou_destino = True
                        break

            if passou_origem and passou_destino:
                distancia = self.calcular_distancia_rota(pontos)
                if distancia > 0.5: 
                    resultados.append({
                        'trip_id': id_rota,
                        'pontos': pontos,
                        'distancia_km': distancia
                    })
            
        return sorted(resultados, key=lambda x: x['distancia_km'])

    def plotar_opcoes(self, origem, destino, max_opcoes=5):
        print(f"Buscando rotas de {origem} para {destino}...")
        
        rotas_analisadas = self.analisar_rotas_com_filtro(origem, destino)
        
        if not rotas_analisadas:
            print("Nenhuma rota faz esse trajeto ou os ônibus estão fora do raio de busca.")
            return folium.Map(location=self.locais[origem], zoom_start=12)

        rotas_analisadas = rotas_analisadas[:max_opcoes]
        melhor_rota = rotas_analisadas[0]

        centro_mapa = (
            (self.locais[origem][0] + self.locais[destino][0]) / 2,
            (self.locais[origem][1] + self.locais[destino][1]) / 2
        )
        mymap = folium.Map(location=centro_mapa, zoom_start=12, tiles='cartodbpositron')

        folium.Marker(self.locais[origem], tooltip=f"Partida: {origem}", icon=folium.Icon(color="green", icon="user")).add_to(mymap)
        folium.Marker(self.locais[destino], tooltip=f"Chegada: {destino}", icon=folium.Icon(color="red", icon="flag")).add_to(mymap)

        print(f"🏆 Melhor rota: {melhor_rota['trip_id']} ({melhor_rota['distancia_km']:.2f} km)")

        cor_idx = 0
        for i, rota in enumerate(rotas_analisadas):
            line_coords = [(p[1], p[0]) for p in rota['pontos']]
            
            if i == 0:
                nome_camada = f"⭐ Principal: {rota['trip_id']} ({rota['distancia_km']:.2f} km)"
                cor, espessura = self.cor_melhor_rota, 7
            else:
                nome_camada = f"Alternativa {i}: {rota['trip_id']} ({rota['distancia_km']:.2f} km)"
                cor = self.cores_alternativas[cor_idx % len(self.cores_alternativas)]
                espessura = 4
                cor_idx += 1

            fg = folium.FeatureGroup(name=nome_camada)
            folium.PolyLine(locations=line_coords, color=cor, weight=espessura, opacity=0.9, tooltip=nome_camada).add_to(fg)
            fg.add_to(mymap)

        folium.LayerControl(collapsed=False).add_to(mymap)
        return mymap


# =====================================================================
# Fazendo a Pesquisa!
# =====================================================================
caminho_arquivo = '/home/maria_araujo/grupo2/.venv/lib/python3.12/site-packages/Palmto_gen/data/rotas.pkl'

recomendador = RecomendadorDeRotas(caminho_arquivo)

mapa_recomendacao = recomendador.plotar_opcoes(origem="Ilha do Governador", destino="Bonsucesso", max_opcoes=5)

mapa_recomendacao.save("recomendacao_rotas.html")
print("✅ Pesquisa concluída e mapa gerado!")